In [ ]:
# 1. Create a new workspace user (innowise@innowise.com) and add them to a new group named
# innowise.

In [2]:
from src.core.services.connection.databricks_connector import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="testing-pipeline")
conn.connect()
spark = conn.spark


2026-10-08 11:43:21.430 | INFO     | src.core.services.connection.databricks_connector:connect:34 - SparkSession ready | app=testing-pipeline | version=4.2.0 | runtime=serverless-connect


In [2]:
spark.sql("GRANT USE CATALOG ON CATALOG yandex_go_dev TO `innowise`")

spark.sql("GRANT USE SCHEMA ON SCHEMA yandex_go_dev.gold TO `innowise`")

spark.sql("GRANT SELECT ON SCHEMA yandex_go_dev.gold TO `innowise`")

grants_df = spark.sql("SHOW GRANTS ON SCHEMA yandex_go_dev.gold").show(truncate=False)
display(grants_df)

+---------+----------+----------+------------------+
|Principal|ActionType|ObjectType|ObjectKey         |
+---------+----------+----------+------------------+
|innowise |SELECT    |SCHEMA    |yandex_go_dev.gold|
|innowise |USE SCHEMA|SCHEMA    |yandex_go_dev.gold|
+---------+----------+----------+------------------+



None

In [3]:

spark.sql("REVOKE SELECT ON SCHEMA yandex_go_dev.gold FROM `innowise`")
spark.sql("REVOKE USE SCHEMA ON SCHEMA yandex_go_dev.gold FROM `innowise`")

display(spark.sql("SHOW GRANTS ON SCHEMA yandex_go_dev.gold").show(truncate=False))

+---------+----------+----------+---------+
|Principal|ActionType|ObjectType|ObjectKey|
+---------+----------+----------+---------+
+---------+----------+----------+---------+



None

In [3]:
from src.core.services.audit.reporting import WorkflowReportService

reporter = WorkflowReportService()
report = reporter.generate_report(spark)

print("=" * 80)
print(
    f"LAKEHOUSE AUDIT REPORT | Environment: {report.environment.upper()} | Status: {report.summary.pipeline_status}"
)
print(
    f"Catalog: {report.catalog} | Total Tables: {report.summary.total_monitored_tables} | Total Records: {report.summary.total_rows_across_lakehouse:,}"
)
print("=" * 80)
print(
    f"{'LAYER':<8} | {'TABLE':<18} | {'ROWS':<10} | {'VERSION':<8} | {'STATUS':<10} | {'LAST MODIFIED'}"
)
print("-" * 80)

for layer, tables in report.layers.items():
    for table_name, meta in tables.items():
        v = f"v{meta.latest_version}" if meta.latest_version is not None else "-"
        mod = meta.last_modified if meta.last_modified else "-"
        print(
            f"{layer.upper():<8} | {table_name:<18} | {meta.row_count:<10,} | {v:<8} | {meta.status.value:<10} | {mod}"
        )

print("=" * 80)

print("\nOUTPUT:\n")
print(report.model_dump_json(indent=2))

2026-10-08 11:43:37.005 | INFO     | src.core.services.audit.reporting:generate_report:77 - Generating Pydantic-validated Audit Report for [yandex_go_dev]...
2026-10-08 11:44:05.145 | INFO     | src.core.services.audit.reporting:generate_report:119 - Audit report successfully saved to: /Volumes/yandex_go_dev/raw_files/landing/reports/workflow_summary.json


LAKEHOUSE AUDIT REPORT | Environment: DEV | Status: SUCCESS
Catalog: yandex_go_dev | Total Tables: 10 | Total Records: 35,415
LAYER    | TABLE              | ROWS       | VERSION  | STATUS     | LAST MODIFIED
--------------------------------------------------------------------------------
BRONZE   | taxi               | 10,000     | v4       | HEALTHY    | 2026-10-05 11:25:50
BRONZE   | drivers            | 250        | v4       | HEALTHY    | 2026-10-03 18:03:19
BRONZE   | users              | 1,500      | v2       | HEALTHY    | 2026-10-05 11:25:40
SILVER   | taxi               | 10,000     | v6       | HEALTHY    | 2026-10-05 12:00:54
SILVER   | drivers            | 250        | v2       | HEALTHY    | 2026-10-03 21:29:56
SILVER   | users              | 1,492      | v4       | HEALTHY    | 2026-10-05 11:27:31
GOLD     | taxi_metrics       | 789        | v17      | HEALTHY    | 2026-10-05 11:43:58
GOLD     | driver_metrics     | 25         | v4       | HEALTHY    | 2026-10-05 10:52:4